# Analog Renaissance — Finance Crack

**Session 21 · monogate v1.4.0**

Key result: every component of the Black-Scholes formula is expressible in the EML/DEML operator family.

| Component | EML form | Nodes | Backend |
|-----------|----------|-------|---------|
| Discount factor `exp(-rT)` | `deml(rT, 1)` | 1 | DEML |
| Log-moneyness `ln(S/K)` | 3-node EML ln | 3 | EML |
| Normal CDF `N(x)` | erf-based | 3 | CBEST |
| Full call price `C` | composition | ~15 | BEST+DEML+CBEST |

In [ ]:
from monogate.finance import (
    black_scholes_call, black_scholes_put,
    bs_call_eml, bs_components_eml,
    bs_discount_cb, bs_log_moneyness_cb,
    bs_d1_cb, bs_d2_cb, FINANCE_CATALOG,
)
import math

# ATM 1-year call: S=K=100, r=5%, sigma=20%
S, K, r, T, sigma = 100.0, 100.0, 0.05, 1.0, 0.20
comp = bs_components_eml(S, K, r, T, sigma)
print(f"Black-Scholes call price: {comp['call_price']:.6f}")
print(f"Discount factor (1-node DEML): {comp['discount']:.6f}  [expected: {math.exp(-r*T):.6f}]")
print(f"Log-moneyness (3-node EML):    {comp['log_moneyness']:.6f}  [ATM → 0]")
print(f"d1: {comp['d1']:.6f}   d2: {comp['d2']:.6f}")
print(f"N(d1): {comp['N_d1']:.6f}   N(d2): {comp['N_d2']:.6f}")

In [ ]:
# Cross-validate: reference vs EML-routed formula
test_cases = [
    (80.0, 100.0, 0.05, 1.0, 0.20),
    (100.0, 100.0, 0.05, 1.0, 0.20),
    (120.0, 100.0, 0.05, 1.0, 0.20),
    (100.0, 100.0, 0.05, 0.5, 0.30),
    (100.0, 100.0, 0.10, 2.0, 0.15),
]

print(f"{'S':>6} {'K':>6} {'r':>5} {'T':>4} {'σ':>5} | {'BS_ref':>10} {'BS_eml':>10} {'diff':>12}")
print("-" * 65)
for args in test_cases:
    ref = black_scholes_call(*args)
    eml = bs_call_eml(*args)
    print(f"{args[0]:>6.1f} {args[1]:>6.1f} {args[2]:>5.2f} {args[3]:>4.1f} {args[4]:>5.2f} | {ref:>10.6f} {eml:>10.6f} {abs(ref-eml):>12.2e}")

In [ ]:
# Finance catalog summary
print("FINANCE_CATALOG entries:")
print(f"{'Name':<25} {'Nodes':>6} {'Backend':<20} {'Notes (truncated)'}")
print("-" * 90)
for name, entry in FINANCE_CATALOG.items():
    note = entry['notes'][:55] + '...' if len(entry['notes']) > 55 else entry['notes']
    print(f"{name:<25} {entry['n_nodes']:>6} {entry['backend']:<20} {note}")

In [ ]:
# Cross-domain analogy: the discount factor tree is shared across three domains
from monogate.frontiers.analog_renaissance import AnalogRenaissance

ar = AnalogRenaissance()
finance_analogies = ar.crack_finance()
print(f"{len(finance_analogies)} finance analogies in the registry:\n")
for a in finance_analogies:
    print(f"  {a.one_liner()}")

In [ ]:
# Put-call parity verification
print("Put-call parity: C - P = S - K*exp(-rT)")
print()
for s_val in [80.0, 90.0, 100.0, 110.0, 120.0]:
    c = black_scholes_call(s_val, K, r, T, sigma)
    p = black_scholes_put(s_val, K, r, T, sigma)
    lhs = c - p
    rhs = s_val - K * bs_discount_cb(r, T)
    print(f"  S={s_val:>5.0f}: C-P={lhs:>8.4f}  S-K*D={rhs:>8.4f}  err={abs(lhs-rhs):.2e}")